# Two T gadgets - proxy

Test two T injection gadgets with proxy for T and S

In [1]:
from tqec.compile.compile import compile_block_graph
from tqec.computation.block_graph import BlockGraph
from tqec.computation.correlation import (
    ConditionalCorrelationSurface,
    CorrelationSurface,
    ZXEdge,
    ZXNode,
)
from tqec.computation.cube import ConditionalLeafCubeKind
from tqec.utils.enums import Basis
from tqec.utils.position import Position3D, Direction3D

In [2]:
def make_t_gadget(
    suffix: str = ""
) -> tuple[BlockGraph, str, str]:
    """
    Make t gadget with proxy T and S cubes, extending in the positive X direction
    """

    port_label = f"t_gadget_{suffix}"

    p   = Position3D(0, 0, 0)
    b1  = Position3D(1, 0, 0)
    t   = Position3D(1, 0, -1)
    r   = Position3D(2, 0, 0)
    b2  = Position3D(2, 1, 0)
    c   = Position3D(2, 1, 1)
    b3  = Position3D(3, 0, 0)
    y   = Position3D(3, 0, -1)

    condition = CorrelationSurface(
            span=frozenset({ZXEdge(ZXNode(p, Basis.Z), ZXNode(b1, Basis.Z))})
        )

    # blockgraph
    g = BlockGraph(f"t_gadget_{suffix}")
    g.add_cube(p,  "P", label=port_label)
    g.add_cube(b1, "XZX")
    g.add_cube(t,  "XZX") # T
    g.add_cube(r,  "ZZX")
    g.add_cube(b2, "ZXX")
    g.add_cube(
        c,
        ConditionalLeafCubeKind.ZXX_ZXZ,
        condition=condition)
    g.add_cube(b3, "XZX")
    g.add_cube(y,  "XZX") # Y

    g.add_pipe(p, b1)
    g.add_pipe(t, b1)
    g.add_pipe(b1, r)
    g.add_pipe(r, b2)
    g.add_pipe(b2, c)
    g.add_pipe(r, b3)
    g.add_pipe(b3, y)

    # observables
    t_es = {
        Basis.X: {(p, b1), (t, b1), (b1, r), (r, b2), (b2, c)},
    }
    f_es = {
        Basis.X: {(p, b1), (t, b1), (b1, r), (r, b3), (y, b3)},
        Basis.Z: {(p, b1), (b1, r), (r, b2), (b2, c),}#(r, b3), (y, b3)}, # include for non-proxy
    }

    def _obs_from_set(s: set):
        return CorrelationSurface(
            span=frozenset({
                ZXEdge(ZXNode(u, basis), ZXNode(v, basis)) for basis, edge in s.items() for u, v in edge
        })
    )

    return g, _obs_from_set(t_es), _obs_from_set(f_es)

In [3]:
g, o_true, o_false = make_t_gadget()
g.view_as_html(pop_faces_at_directions=("-Y","+X"), show_correlation_surface=o_false)

## Stack two gadgets

Stack two T-gadgets along z (at `dz=1` and `dz=4`), joined by a vertical spine of cubes/pipes with open ports at `z=0` and `z=5`. Then build a 4-resolution `ConditionalCorrelationSurface` connecting both gadgets' `o_true`/`o_false` observables through the spine (flat-XOR decomposable).

In [4]:
import itertools

GADGET1_DZ, GADGET2_DZ = 1, 4


def _spine_pos(z):
    return Position3D(0, 0, z)


def stack_two_gadgets() -> BlockGraph:
    """Stack two gadgets (dz=1, dz=4) on a vertical spine; open ports at z=0 and z=5."""
    stacked = BlockGraph("stacked_t_gadgets")

    # gadget bodies (skip their ports; the spine owns the x=0,y=0 column)
    for dz in (GADGET1_DZ, GADGET2_DZ):
        gad = make_t_gadget(dz)[0].shift_by(dz=dz)
        for cube in gad.cubes:
            if cube.is_port:
                continue
            stacked.add_cube(cube.position, cube.kind, cube.label, cube.condition)
        for pipe in gad.pipes:
            u, v = pipe.u.position, pipe.v.position
            if u not in stacked or v not in stacked:
                continue  # port pipe (p->b1) added after the spine cube exists
            stacked.add_pipe(u, v, pipe.kind)

    # spine cubes
    stacked.add_cube(_spine_pos(0), "P", label="Input")
    stacked.add_cube(_spine_pos(1), "XZX")  # filled gadget-1 port
    stacked.add_cube(_spine_pos(2), "XZX")
    stacked.add_cube(_spine_pos(3), "XZX")
    stacked.add_cube(_spine_pos(4), "XZX")  # filled gadget-2 port
    stacked.add_cube(_spine_pos(5), "P", label="Output")

    # port -> b1 pipes (spine cube now exists at the port position)
    stacked.add_pipe(_spine_pos(1), Position3D(1, 0, 1))
    stacked.add_pipe(_spine_pos(4), Position3D(1, 0, 4))

    # vertical spine pipes
    for z in range(5):
        stacked.add_pipe(_spine_pos(z), _spine_pos(z + 1))

    return stacked

In [5]:
SPINE_BOTTOM = 0  # input port
SPINE_TOP = 5     # output port


def _spine_edges(z_from, z_to, basis):
    """Edges of the given basis along the spine between z_from and z_to (x=0,y=0)."""
    lo, hi = sorted((z_from, z_to))
    return frozenset(
        ZXEdge(ZXNode(_spine_pos(z), basis), ZXNode(_spine_pos(z + 1), basis))
        for z in range(lo, hi)
    )


def _spine_x_edges():
    """X membrane spanning the full spine (both ports): z=0..5 at x=0,y=0."""
    return _spine_edges(SPINE_BOTTOM, SPINE_TOP, Basis.X)


def build_conditional_observable() -> ConditionalCorrelationSurface:
    """4-resolution observable: each key picks o_true/o_false per gadget, joined by the spine."""
    _, o_true_1, o_false_1 = make_t_gadget(GADGET1_DZ)
    _, o_true_2, o_false_2 = make_t_gadget(GADGET2_DZ)

    # (false-arm span, true-arm span) per gadget. The X leg is carried by the
    # shared full-spine X membrane (below). o_false additionally has a Z leg,
    # routed down the spine to the input port at z=0.
    arms = {
        GADGET1_DZ: (
            o_false_1.shift_by(dz=GADGET1_DZ).span
            | _spine_edges(SPINE_BOTTOM, GADGET1_DZ, Basis.Z),
            o_true_1.shift_by(dz=GADGET1_DZ).span,
        ),
        GADGET2_DZ: (
            o_false_2.shift_by(dz=GADGET2_DZ).span
            | _spine_edges(SPINE_BOTTOM, GADGET2_DZ, Basis.Z),
            o_true_2.shift_by(dz=GADGET2_DZ).span,
        ),
    }
    spine = _spine_x_edges()

    # Combine by symmetric difference (Z2 composition): gadget-internal edges are
    # disjoint (behaves like union), while the two gadgets' Z-to-z0 spine segments
    # overlap on z0..z1 and cancel when both are false -- the only parity-valid way
    # to route both Z legs down the single spine column (avoids the 3-leg Z
    # junction at the filled port (0,0,1)).
    resolutions = {}
    for b0, b1 in itertools.product((False, True), repeat=2):
        span = arms[GADGET1_DZ][b0] ^ arms[GADGET2_DZ][b1] ^ spine
        resolutions[(b0, b1)] = CorrelationSurface(span=frozenset(span))

    def _condition(dz):
        edge = ZXEdge(ZXNode(_spine_pos(dz), Basis.Z), ZXNode(Position3D(1, 0, dz), Basis.Z))
        return CorrelationSurface(span=frozenset({edge}))

    return ConditionalCorrelationSurface(
        conditions=(_condition(GADGET1_DZ), _condition(GADGET2_DZ)),
        resolutions=resolutions,
    )

In [12]:
stacked = stack_two_gadgets()
cco = build_conditional_observable()

# visualize one resolution
stacked.view_as_html(pop_faces_at_directions=("-Y",), show_correlation_surface=cco.resolutions[(True, True)])